## TASK 5 

In [0]:
dbutils.widgets.text("catalog" , "digital_banking")
catalog = dbutils.widgets.get("catalog")

In [0]:
# SCD Type 1 Implementation for Customer Data
# This merges incoming customer changes into silver_customers
# Columns tracked: email, phone, address, city, state, postal_code

# Source and target tables
source_table = f"{catalog}.bronze.bronze_customers" # as we do not have any source data by default taking bronze (which is not cleaned)
target_table = f"{catalog}.silver.silver_customers"

# MERGE statement for SCD Type 1
merge_query = f"""
MERGE INTO {target_table} AS target
USING (
  SELECT 
    customer_id,
    first_name,
    last_name,
    date_of_birth,
    email,
    phone,
    address,
    city,
    state,
    postal_code,
    customer_segment,
    customer_status,
    registration_date,
    CURRENT_TIMESTAMP() AS updated_at
  FROM {source_table}
) AS source
ON target.customer_id = source.customer_id

-- WHEN MATCHED: Update existing records (SCD Type 1 - overwrite changes)
WHEN MATCHED AND (
  -- Only update if tracked columns have changed
  COALESCE(target.email, '') != COALESCE(source.email, '') OR
  COALESCE(target.phone, '') != COALESCE(source.phone, '') OR
  COALESCE(target.address, '') != COALESCE(source.address, '') OR
  COALESCE(target.city, '') != COALESCE(source.city, '') OR
  COALESCE(target.state, '') != COALESCE(source.state, '') OR
  COALESCE(target.postal_code, '') != COALESCE(source.postal_code, '')
) THEN UPDATE SET
  target.first_name = source.first_name,
  target.last_name = source.last_name,
  target.date_of_birth = source.date_of_birth,
  target.email = source.email,
  target.phone = source.phone,
  target.address = source.address,
  target.city = source.city,
  target.state = source.state,
  target.postal_code = source.postal_code,
  target.customer_segment = source.customer_segment,
  target.customer_status = source.customer_status,
  target.updated_at = source.updated_at

-- WHEN NOT MATCHED: Insert new customer records
WHEN NOT MATCHED THEN INSERT (
  customer_id,
  first_name,
  last_name,
  date_of_birth,
  email,
  phone,
  address,
  city,
  state,
  postal_code,
  customer_segment,
  customer_status,
  registration_date,
  updated_at
) VALUES (
  source.customer_id,
  source.first_name,
  source.last_name,
  source.date_of_birth,
  source.email,
  source.phone,
  source.address,
  source.city,
  source.state,
  source.postal_code,
  source.customer_segment,
  source.customer_status,
  source.registration_date,
  source.updated_at
)
"""

# Execute the MERGE
spark.sql(merge_query)

print(f"Target table: {target_table}")